# 使用 Zilliz 和 OpenAI 进行过滤搜索
### 寻找你的下一部电影

在本 notebook 中，我们将使用 OpenAI 为电影简介生成嵌入，并在 Zilliz 中利用这些嵌入查找相关电影。为了缩小搜索结果并尝试一些新东西，我们将使用过滤功能进行元数据搜索。本示例中的数据集来自 HuggingFace datasets，包含 8 千多条电影记录。

首先，我们来下载本 notebook 所需的库：
- `openai` 用于与 OpenAI 嵌入服务通信
- `pymilvus` 用于与 Zilliz 服务器通信
- `datasets` 用于下载数据集
- `tqdm` 用于显示进度条


In [ ]:
! pip install openai pymilvus datasets tqdm

要让 Zilliz 运行起来，请参阅[这里](https://zilliz.com/doc/quick_start)。设置好账户和数据库后，接着设置以下值：
- URI：数据库运行所在的 URI
- USER：数据库用户名
- PASSWORD：数据库密码
- COLLECTION_NAME：Zilliz 中集合的名称
- DIMENSION：嵌入的维度
- OPENAI_ENGINE：要使用的嵌入模型
- openai.api_key：你的 OpenAI 账户密钥
- INDEX_PARAM：集合使用的索引设置
- QUERY_PARAM：要使用的搜索参数
- BATCH_SIZE：一次嵌入并插入多少条文本

In [1]:
import openai

URI = 'your_uri'
TOKEN = 'your_token' # TOKEN == user:password or api_key
COLLECTION_NAME = 'book_search'
DIMENSION = 1536
OPENAI_ENGINE = 'text-embedding-3-small'
openai.api_key = 'sk-your_key'

INDEX_PARAM = {
    'metric_type':'L2',
    'index_type':"AUTOINDEX",
    'params':{}
}

QUERY_PARAM = {
    "metric_type": "L2",
    "params": {},
}

BATCH_SIZE = 1000

In [3]:
from pymilvus import connections, utility, FieldSchema, Collection, CollectionSchema, DataType

# Connect to Zilliz Database
connections.connect(uri=URI, token=TOKEN)

In [4]:
# Remove collection if it already exists
if utility.has_collection(COLLECTION_NAME):
    utility.drop_collection(COLLECTION_NAME)

In [5]:
# Create collection which includes the id, title, and embedding.
fields = [
    FieldSchema(name='id', dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name='title', dtype=DataType.VARCHAR, max_length=64000),
    FieldSchema(name='type', dtype=DataType.VARCHAR, max_length=64000),
    FieldSchema(name='release_year', dtype=DataType.INT64),
    FieldSchema(name='rating', dtype=DataType.VARCHAR, max_length=64000),
    FieldSchema(name='description', dtype=DataType.VARCHAR, max_length=64000),
    FieldSchema(name='embedding', dtype=DataType.FLOAT_VECTOR, dim=DIMENSION)
]
schema = CollectionSchema(fields=fields)
collection = Collection(name=COLLECTION_NAME, schema=schema)

In [6]:
# Create the index on the collection and load it.
collection.create_index(field_name="embedding", index_params=INDEX_PARAM)
collection.load()

## 数据集
Zilliz 运行起来后，就可以开始获取数据了。`Hugging Face Datasets` 是一个汇集了大量用户数据集的平台，本示例使用的是 HuggingLearners 的 netflix-shows 数据集。该数据集包含 8 千多部电影及其元数据。我们将对每条简介生成嵌入，并将其与标题、类型、上映年份和评级一起存入 Zilliz。

In [7]:
import datasets

# Download the dataset 
dataset = datasets.load_dataset('hugginglearners/netflix-shows', split='train')

/Users/filiphaltmayer/miniconda3/envs/haystack/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Found cached dataset csv (/Users/filiphaltmayer/.cache/huggingface/datasets/hugginglearners___csv/hugginglearners--netflix-shows-03475319fc65a05a/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317)


## 插入数据
数据已经下载到本地，现在可以开始生成嵌入并插入 Zilliz 了。嵌入函数接收文本，并以列表形式返回嵌入。

In [8]:
# Simple function that converts the texts to embeddings
def embed(texts):
    embeddings = openai.Embedding.create(
        input=texts,
        engine=OPENAI_ENGINE
    )
    return [x['embedding'] for x in embeddings['data']]


下一步执行实际的插入操作。我们遍历所有条目并组成批次，每当达到设定的批大小时就插入一次。循环结束后，如果还有剩余的批次，就将其插入。

In [9]:
from tqdm import tqdm

data = [
    [], # title
    [], # type
    [], # release_year
    [], # rating
    [], # description
]

# Embed and insert in batches
for i in tqdm(range(0, len(dataset))):
    data[0].append(dataset[i]['title'] or '')
    data[1].append(dataset[i]['type'] or '')
    data[2].append(dataset[i]['release_year'] or -1)
    data[3].append(dataset[i]['rating'] or '')
    data[4].append(dataset[i]['description'] or '')
    if len(data[0]) % BATCH_SIZE == 0:
        data.append(embed(data[4]))
        collection.insert(data)
        data = [[],[],[],[],[]]

# Embed and insert the remainder 
if len(data[0]) != 0:
    data.append(embed(data[4]))
    collection.insert(data)
    data = [[],[],[],[],[]]


100%|██████████| 8807/8807 [00:54<00:00, 162.59it/s]


## 查询数据库
数据已经安全地插入 Zilliz，现在可以执行查询了。查询接收一个元组，包含你要搜索的电影描述以及要使用的过滤条件。关于过滤条件的更多信息请参阅[这里](https://milvus.io/docs/boolean.md)。搜索会先打印出你的描述和过滤表达式，然后针对每个结果打印电影的得分、标题、类型、上映年份、评级和简介。

In [10]:
import textwrap

def query(query, top_k = 5):
    text, expr = query
    res = collection.search(embed(text), anns_field='embedding', expr = expr, param=QUERY_PARAM, limit = top_k, output_fields=['title', 'type', 'release_year', 'rating', 'description'])
    for i, hit in enumerate(res):
        print('Description:', text, 'Expression:', expr)
        print('Results:')
        for ii, hits in enumerate(hit):
            print('\t' + 'Rank:', ii + 1, 'Score:', hits.score, 'Title:', hits.entity.get('title'))
            print('\t\t' + 'Type:', hits.entity.get('type'), 'Release Year:', hits.entity.get('release_year'), 'Rating:', hits.entity.get('rating'))
            print(textwrap.fill(hits.entity.get('description'), 88))
            print()

my_query = ('movie about a fluffly animal', 'release_year < 2019 and rating like \"PG%\"')

query(my_query)

Description: movie about a fluffly animal Expression: release_year < 2019 and rating like "PG%"
Results:
	Rank: 1 Score: 0.30085673928260803 Title: The Lamb
		Type: Movie Release Year: 2017 Rating: PG
A big-dreaming donkey escapes his menial existence and befriends some free-spirited
animal pals in this imaginative retelling of the Nativity Story.

	Rank: 2 Score: 0.3352621793746948 Title: Puss in Boots
		Type: Movie Release Year: 2011 Rating: PG
The fabled feline heads to the Land of Giants with friends Humpty Dumpty and Kitty
Softpaws on a quest to nab its greatest treasure: the Golden Goose.

	Rank: 3 Score: 0.3415083587169647 Title: Show Dogs
		Type: Movie Release Year: 2018 Rating: PG
A rough and tough police dog must go undercover with an FBI agent as a prim and proper
pet at a dog show to save a baby panda from an illegal sale.

	Rank: 4 Score: 0.3428957462310791 Title: Open Season 2
		Type: Movie Release Year: 2008 Rating: PG
Elliot the buck and his forest-dwelling cohorts must